# Setting up a basic Agent with Google ADK

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Needs MOONSHOT_API_KEY and SERPAPI_API_KEY in .env

True

In [2]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

In [3]:
from datetime import datetime

today = datetime.today().strftime("%B ") + str(datetime.today().day) + ("th" if 11<=datetime.today().day<=13 else {1:"st",2:"nd",3:"rd"}.get(datetime.today().day%10,"th")) + datetime.today().strftime(", %Y")
print(today)

May 6th, 2026


In [ ]:
model = LiteLlm(
    model='moonshot/kimi-k2',
    api_key=os.getenv('MOONSHOT_API_KEY'),
)

In [11]:
from langchain_community.utilities import SerpAPIWrapper

serpapi = SerpAPIWrapper(serpapi_api_key=os.getenv('SERPAPI_API_KEY'))

def search(query: str) -> str:
    """Search the web for real-time information. Use this for current events, weather, sports, news, etc."""
    return serpapi.run(query)

# Quick test
search('weather in San Francisco')

"{'type': 'weather_result', 'temperature': '56', 'unit': 'Fahrenheit', 'precipitation': '5%', 'humidity': '87%', 'wind': '4 mph', 'location': 'San Francisco, CA', 'date': 'Wednesday 8:00 AM', 'weather': 'Partly sunny'}"

In [12]:
search_agent = Agent(
    name='search_agent',
    model=model,
    description='Agent that answers questions using web search.',
    instruction=f'Today is {today}. You are a helpful assistant. Use the search tool to answer questions about current events.',
    tools=[search]
)

In [13]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

session_service = InMemorySessionService()

runner = Runner(
    agent=search_agent,
    app_name='search_app',
    session_service=session_service
)

In [14]:
async def ask(query: str, user_id='user1', session_id='session1'):
    """Send a query to the agent and return the final response text."""
    session = await session_service.get_session(
        app_name='search_app', user_id=user_id, session_id=session_id
    )
    if not session:
        await session_service.create_session(
            app_name='search_app', user_id=user_id, session_id=session_id
        )
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=query)]
    )
    final_response = ''
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

In [15]:
response = await ask('Who is the current Ravens QB?')
print(response)

The Baltimore Ravens’ current starting quarterback is **Lamar Jackson**.


In [16]:
response = await ask('What is the weather in San Francisco right now?')
print(response)

In San Francisco right now, it’s **sunny** and about **63°F**.

Details:
- **Precipitation:** 5%
- **Humidity:** 80%
- **Wind:** 14 mph


## Custom Tools

In [17]:
def execute_python_code(code: str) -> str:
    """Execute a Python expression and return the result as a string."""
    try:
        result = eval(code)
        return f'Result: {result}'
    except Exception as e:
        return f'Error: {e}'


code_agent = Agent(
    name='code_agent',
    model=model,
    description='Agent that can execute Python code.',
    instruction='You are a helpful assistant. Use the execute_python_code tool to run Python expressions when asked math or coding questions.',
    tools=[execute_python_code]
)

In [18]:
code_runner = Runner(
    agent=code_agent,
    app_name='code_app',
    session_service=session_service
)


async def ask_code(query: str, user_id='user1', session_id='code_session1'):
    """Send a query to the code agent and return the final response text."""
    session = await session_service.get_session(
        app_name='code_app', user_id=user_id, session_id=session_id
    )
    if not session:
        await session_service.create_session(
            app_name='code_app', user_id=user_id, session_id=session_id
        )
    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=query)]
    )
    final_response = ''
    async for event in code_runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

In [19]:
response = await ask_code('What is 25 * 37 + 100?')
print(response)

1025


In [20]:
response = await ask_code('What is the 20th Fibonacci number?')
print(response)

The 20th Fibonacci number is 6765.


## Short Term Memory

ADK sessions provide built-in short term memory - messages within the same session are remembered.

In [21]:
response = await ask_code('Hi, my name is Sinan', session_id='memory_test')
print(response)

Hi Sinan — nice to meet you! How can I help?


In [22]:
response = await ask_code('What is my name?', session_id='memory_test')
print(response)

Your name is Sinan.


In [23]:
# Different session = no memory of previous conversation
response = await ask_code('What is my name?', session_id='different_session')
print(response)

I don’t know your name unless you tell me.
